# Quantifying the what-if melt flattening, at basin scale

In [ ]:
# basic
import numpy as np
import pandas as pd

# spatial
import xarray as xr

# plotting
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler

from src.utils import load_config, assign_region
from src.plotting import base_layout, scenario_colors, region_symbols, save_figure

cfg = load_config()
scenarios_names = [s["name"] for s in cfg["scenarios"]]
scenario_ids = [s["id"] for s in cfg["scenarios"]]
colors = scenario_colors(cfg)
symbols = region_symbols(cfg)

## Data

In [ ]:
metadata = pd.read_csv("data/glacier/glacier_metadata_evolution.csv", index_col=0)[["cenlat", "cenlon", "inv_volume_km3"]]

df_historical = xr.open_dataset("data/glacier/glacier_evolution_2000_2024_glacier.nc")
df_future     = xr.open_dataset("data/glacier/glacier_evolution_2025_2099_glacier.nc")

# assign each glacier to a region from its centroid latitude, then collapse the glacier-specific
region = xr.DataArray(metadata["cenlat"].apply(assign_region, regions=cfg["regions"]),
                       dims="rgi_id", coords={"rgi_id": metadata.index}, name="region")
df_historical = df_historical.groupby(region).sum()
df_future     = df_future.groupby(region).sum()

# --- melt: historical region average vs. each ensemble member's future average ---
ref_start, ref_end = int(cfg["time"]["ref_start"][:4]), int(cfg["time"]["ref_end"][:4])
melt_hist = df_historical["melt_on_glacier"].sel(time=slice(ref_start, ref_end)).mean("time").to_pandas()
melt_hist.name = "historical"

melt_future = df_future["melt_on_glacier"].mean("time").transpose("region", "warming_level").to_pandas()
member_count = {}
member_cols = []
for wl in melt_future.columns:
    member_cols.append(f"{wl}_{member_count.get(wl, 0)}")
    member_count[wl] = member_count.get(wl, 0) + 1
melt_future.columns = member_cols

melt = pd.concat([melt_hist, melt_future], axis=1)

# --- volume: 2020 region volume vs. each ensemble member's 2100 volume, by scenario ---
volume_hist = df_historical["volume"].sel(time=2020).to_pandas()
volume_future = df_future["volume"].sel(time=2100).transpose("region", "warming_level").to_pandas()
volume_future.columns = member_cols
volume = 100 * (1 - volume_future.div(volume_hist, axis=0))

In [ ]:
region_df = []

for name, sid in zip(scenarios_names, scenario_ids):
    member_cols = [c for c in melt.columns if c.startswith(f"{sid}_")]

    for mc in member_cols:
        melt_change = 100 * ((melt[mc] / melt["historical"]) - 1)

        df = pd.DataFrame({
            "hist_avg_melt": melt["historical"],
            "melt_change": melt_change,
            "volume_loss": volume[mc],
        })

        df["scenario"] = name
        df["region"] = df.index
        region_df.append(df)

region_df = pd.concat(region_df).dropna(subset=["volume_loss", "melt_change"])

## Plot

In [ ]:
fig = go.Figure()

# --- basin cloud, volume loss vs melt reduction, one point per region per ensemble member ---
fig.add_trace(go.Scatter(x=[-20, 105], y=[20, -105], mode="lines", line=dict(color="grey", width=1, dash="dot"),
                          showlegend=False))

for i, wl in enumerate(scenarios_names):
    sub = region_df[region_df.scenario == wl]
    fig.add_trace(go.Scatter(
        x=sub["volume_loss"], y=sub["melt_change"], mode="markers", name=wl,  opacity=0.8,
        marker=dict(size=15, color=colors[i], symbol=sub["region"].map(symbols),
                    line=dict(width=0.1, color="black")),
        showlegend=True, legendgroup=wl,
        legendgrouptitle_text="Scenario:" if i == 0 else None,
    ))

# --- region legend, symbol only (color already carries scenario) ---
for j, (region, symbol) in enumerate(symbols.items()):
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers", name=region,
        marker=dict(size=11, symbol=symbol, color="#767676", line=dict(width=0)),
        showlegend=True, legendgroup="region",
        legendgrouptitle_text="Region:" if j == 0 else None,
    ))

fig.update_xaxes(title_text="Glacier volume loss (2020 vs. 2100)", title_standoff=5, ticksuffix="%", range=[-5, 100])
fig.update_yaxes(title_text="Glacier melt change (historical vs. what-if)", ticksuffix="%", title_standoff=5, range=[-105, 20])

fig.update_layout(**base_layout(
    width=700, height=500,
    margin=dict(l=80, r=170, b=50, t=20),
    legend=dict(orientation="v", y=1, x=1.02, xanchor="left", yanchor="top", tracegroupgap=10),
))

save_figure(fig, "figures/fig_whatif_melt_flattening.svg", width=700, height=500,)
fig.show()